In [11]:
import numpy as np
import sys
import types
import os
# =========================================================
# 0. HARD DISABLE MPI (ROBUST VERSION)
# =========================================================
# import os
# import sys
# import types

# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["DESI_DISABLE_MPI"] = "1"


# # -------------------------
# # Fake MPI implementation
# # -------------------------
# class FakeComm:
#     def Get_rank(self): return 0
#     def Get_size(self): return 1
#     def Barrier(self): pass
#     def bcast(self, x, root=0): return x
#     def allreduce(self, x, op=None): return x
#     def gather(self, x, root=0): return [x]
#     def allgather(self, x): return [x]


# class FakeStatus:
#     pass


# class FakeMPI:
#     COMM_WORLD = FakeComm()
#     COMM_SELF = FakeComm()

#     ANY_SOURCE = -1
#     ANY_TAG = -1

#     SUM = None
#     MAX = None
#     MIN = None

#     Status = FakeStatus

#     @staticmethod
#     def Get_processor_name():
#         return "fake-node"


# # -------------------------
# # Fake mpi4py module tree
# # -------------------------
# mpi4py = types.ModuleType("mpi4py")
# mpi4py.MPI = FakeMPI

# sys.modules["mpi4py"] = mpi4py
# sys.modules["mpi4py.MPI"] = FakeMPI



# ---------------------------
# NOW safe to import desilike
# ---------------------------
from desilike.theories.galaxy_clustering import DampedBAOWigglesTracerPowerSpectrumMultipoles
from desilike.observables.galaxy_clustering import TracerPowerSpectrumMultipolesObservable
from desilike.likelihoods import ObservablesGaussianLikelihood
from desilike.profilers import MinuitProfiler


# ---------------------------
# Load data
# ---------------------------
filename = "../data/monopoles/box/mag=-21.2_sep=1.0-150.0_binsep=2.0_full.npz"

data = np.load(filename)
print("keys:", data.files)

s = data["s"]
xi = data["xi0"]


# ---------------------------
# Fake covariance (temporary)
# ---------------------------
sigma_xi = 0.05 * np.abs(xi)
cov = np.diag(sigma_xi**2)


# ---------------------------
# Theory
# ---------------------------
theory = TracerPowerSpectrumMultipolesObservable()

#theory.init.update(
#    k=s
#)

params = theory.params
params['alpha_iso'].update(value=1.0, prior={'limits': [0.8, 1.2]})
params['sigma_nl'].update(value=5.0, prior={'limits': [0., 20.]})

# broadband
for name in params.select(basename='a*'):
    params[name].update(fixed=False)


# ---------------------------
# Likelihood
# ---------------------------
observable = TracerPowerSpectrumMultipolesObservable(data=xi, covariance=cov, theory=theory)
likelihood = ObservablesGaussianLikelihood(observables=[observable])


# ---------------------------
# Fit
# ---------------------------
profiler = MinuitProfiler(likelihood)
bestfit = profiler.maximize()


print("\nalpha =", bestfit.params['alpha'])
print("Sigma_nl =", bestfit.params['Sigma_nl'])
print("chi2 =", bestfit.chi2)

keys: ['s', 'xi0']


KeyError: 'Parameter alpha_iso not found'

In [12]:
theory.all_params

PipelineError: Error in method initialize of <desilike.observables.galaxy_clustering.power_spectrum.TracerPowerSpectrumMultipolesObservable object at 0x79b5e9674f90>

### Second Try

In [16]:
import numpy as np

from desilike.theories.galaxy_clustering import DampedBAOWigglesTracerCorrelationFunctionMultipoles
from desilike.likelihoods import ObservablesGaussianLikelihood
from desilike.profilers import MinuitProfiler


# ---------------------------
# 1. Load your data
# ---------------------------
filename = "../data/monopoles/box/mag=-21.2_sep=1.0-150.0_binsep=2.0_full.npz"

data = np.load(filename)
print("keys:", data.files)

s = data["s"]
xi = data["xi0"]


# ---------------------------
# 2. Temporary covariance
# ---------------------------
# Since you don't have jackknife yet,
# we assume uncorrelated errors with 5% relative uncertainty.

sigma_xi = 0.05 * np.abs(xi)
cov = np.diag(sigma_xi**2)


# ---------------------------
# 3. Define BAO theory
# ---------------------------
theory = DampedBAOWigglesTracerCorrelationFunctionMultipoles()

theory.init.update(
    mode='monopole',
    s=s,
    reconstruction=False
)


# ---------------------------
# 4. Parameter setup
# ---------------------------
params = theory.params

# BAO scale
params['alpha'].update(value=1.0, prior={'limits': [0.8, 1.2]})

# Nonlinear damping
params['Sigma_nl'].update(value=5.0, prior={'limits': [0., 20.]})

# Bias / amplitude
if 'b1' in params.names():
    params['b1'].update(value=1.5)

# Broadband nuisance terms (if present)
for name in params.select(basename='a*'):
    params[name].update(fixed=False)


# ---------------------------
# 5. Build likelihood
# ---------------------------
observable = xi

likelihood = ObservablesGaussianLikelihood(observables=[observable])

#z = 1.
#template = BAOPowerSpectrumTemplate(z=z, fiducial='DESI')
#theory = FlexibleBAOWigglesTracerPowerSpectrumMultipoles(template=template, broadband='pcs', wiggles='pcs')
#params = {'b1': 2.}
# Generate synthetic monopole and quadrupole, between 0.02 and 0.35 h/Mpc
#observable = TracerPowerSpectrumMultipolesObservable(data=params, covariance=None,
#                                                     klim={0: [0.005, 0.35, 0.005], 2: [0.005, 0.35, 0.005]},
#                                                     theory=theory)

#observable.init.update(data=observable.flatdata)  # fix the data vector

from desilike.observables.galaxy_clustering import BoxFootprint, ObservablesCovarianceMatrix

footprint = BoxFootprint(volume=1e9, nbar=.0024)  # box with volume of 5 (Gpc/h)^3 and density of 1e-4 (h/Mpc)^3
covariance = ObservablesCovarianceMatrix(observables=[observable], footprints=[footprint])
likelihood = ObservablesGaussianLikelihood(observables=[observable], covariance=covariance(**params))

# ---------------------------
# 6. Fit (maximum likelihood)
# ---------------------------
profiler = MinuitProfiler(likelihood)

bestfit = profiler.maximize()


# ---------------------------
# 7. Results
# ---------------------------
print("\n===== BEST FIT =====")

print("alpha =", bestfit.params['alpha'])
print("Sigma_nl =", bestfit.params.get('Sigma_nl', None))

print("chi2 =", bestfit.chi2)

keys: ['s', 'xi0']


KeyError: 'Parameter alpha not found'

In [19]:
theory.params


ParameterCollection(['b1', 'dbeta', 'sigmas', 'sigmapar', 'sigmaper', 'al0_-2', 'al0_-1', 'al0_0', 'al0_1', 'al0_2', 'al2_-2', 'al2_-1', 'al2_0', 'al2_1', 'al2_2', 'al4_-2', 'al4_-1', 'al4_0', 'al4_1', 'al4_2'])